# 🚧 IRIS — YOLOv8 Pothole Detection Fine-Tuning
**Team Grey Hats | Technomax 2026**

This notebook fine-tunes your existing `best.pt` on a Roboflow pothole dataset.

> ⚠️ Before running: **Runtime → Change runtime type → T4 GPU → Save**

## Step 1 — Install Dependencies

In [ ]:
!pip install ultralytics roboflow -q
import ultralytics
ultralytics.checks()

## Step 2 — Upload your existing best.pt
Run this cell to upload `best.pt` from your laptop into Colab.

In [ ]:
from google.colab import files
uploaded = files.upload()  # select best.pt from D:\Btech Projects\IRIS\models\
print('Uploaded:', list(uploaded.keys()))

## Step 3 — Download Pothole Dataset from Roboflow
Go to roboflow.com → search 'pothole' → pick dataset → Versions → YOLOv8 → Show download code → paste API key below.

In [ ]:
from roboflow import Roboflow

rf = Roboflow(api_key="YOUR_ROBOFLOW_API_KEY")  # <-- replace
project = rf.workspace("YOUR_WORKSPACE").project("YOUR_PROJECT")
version = project.version(1)
dataset = version.download("yolov8")
print(f"Dataset: {dataset.location}")

## Step 4 — Fine-tune from existing best.pt

In [ ]:
from ultralytics import YOLO

# Start from YOUR existing trained model, not scratch
model = YOLO('/content/best.pt')

results = model.train(
    data=f"{dataset.location}/data.yaml",
    epochs=50,
    imgsz=640,
    batch=16,
    name='iris_pothole_v2',
    patience=10,
    device=0,
    workers=2,
    exist_ok=True
)
print('Training complete!')

## Step 5 — Evaluate the Model

In [ ]:
metrics = model.val()
print(f"mAP50:     {metrics.box.map50:.3f}")
print(f"mAP50-95:  {metrics.box.map:.3f}")
print(f"Precision: {metrics.box.mp:.3f}")
print(f"Recall:    {metrics.box.mr:.3f}")

## Step 6 — Preview on Test Images

In [ ]:
import glob
from IPython.display import Image, display

test_imgs = glob.glob(f"{dataset.location}/test/images/*.jpg")[:4]
model.predict(test_imgs, conf=0.45, save=True, project='/content/preview', exist_ok=True)

for img in glob.glob('/content/preview/predict/*.jpg'):
    display(Image(img, width=600))

## Step 7 — Plot Training Curves

In [ ]:
from IPython.display import Image, display
import os
d = 'runs/detect/iris_pothole_v2'
for f in ['results.png', 'confusion_matrix.png', 'PR_curve.png']:
    p = os.path.join(d, f)
    if os.path.exists(p):
        print(f)
        display(Image(p, width=700))

## Step 8 — Download best.pt to your laptop

In [ ]:
from google.colab import files
import shutil

best = 'runs/detect/iris_pothole_v2/weights/best.pt'
shutil.copy(best, '/content/best.pt')
files.download('/content/best.pt')
print('Done! Replace D:/Btech Projects/IRIS/models/best.pt with this file.')

## ✅ Done!

After downloading:
1. Replace `D:\Btech Projects\IRIS\models\best.pt`
2. Run `python main.py` — IRIS now uses the improved model

### 📊 Target metrics:
| Metric | Good | Excellent |
|--------|------|-----------|
| mAP50 | > 0.70 | > 0.85 |
| Precision | > 0.75 | > 0.90 |
| Recall | > 0.70 | > 0.85 |

### 🔧 If accuracy is low:
- Increase `epochs` to 100
- Try `yolov8s.pt` instead of nano
- Lower `CONFIDENCE_THRESHOLD` in `config.py` to 0.35